In [4]:
import sys
from pathlib import Path

# Notebook is in /notebooks → project root is one level up
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Added to sys.path:", PROJECT_ROOT)



Added to sys.path: c:\Users\fatem\OneDrive\Desktop\capstone\Agentic-Crypto-Return-Service


In [5]:
from core.models.horizon_scenarios import forecast_horizon
print("Imported forecast_horizon successfully")

Imported forecast_horizon successfully


In [7]:
import numpy as np
import pandas as pd

from core.models import probabilistic_quantile as pq
import core.models.horizon_scenarios as hs


In [8]:
# Load engineered features (this file already exists from earlier steps)
df = pq.load_features_csv("../data/processed/features/BTC_features.csv")

print("Loaded df with rows:", len(df))
print("Columns:", df.columns.tolist())
print("Date range:", df["date"].min(), "→", df["date"].max())


Loaded df with rows: 2201
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'ticker', 'market_cap', 'log_ret_1d', 'log_ret_5d', 'log_ret_10d', 'vol_7d', 'vol_30d', 'risk_adj_ret_1d', 'vol_ratio_7d_30d', 'drawdown_30d']
Date range: 2020-01-31 00:00:00 → 2026-02-09 00:00:00


Load the features dataframe you’ll use for horizon queries

In [9]:
import core.models.horizon_scenarios as hs

features_df = df.copy()   # must contain a 'date' column and your feature columns
print("features_df rows:", len(features_df))
print("last available date:", features_df["date"].max())


features_df rows: 2201
last available date: 2026-02-09 00:00:00


In [11]:
FEATURES = [
    "log_ret_1d",
    "log_ret_5d",
    "log_ret_10d",
    "vol_7d",
    "vol_30d",
    "risk_adj_ret_1d",
    "vol_ratio_7d_30d",
    "drawdown_30d",
]

df_model = pq.add_next_day_target(df)
train, test = pq.time_split(df_model, train_frac=0.8)

bundle = pq.fit_quantile_models(
    train_df=train,
    feature_cols=FEATURES,
    quantiles=[0.01,0.05,0.10,0.25,0.50,0.75,0.90,0.95,0.99],
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
)

print("Model bundle ready.")


Model bundle ready.


Example A — “few weeks” using horizon_days

Let’s simulate 10 trading days (~2 weeks).

In [12]:
start_date = features_df["date"].max()  # most recent row date

out_2w = hs.forecast_horizon(
    bundle=bundle,
    features_df=features_df,
    start_date=start_date,
    horizon_days=10,
    n_scenarios=5000,
    alpha=0.05,
    seed=42,
)

out_2w.summary


{'median': -0.04826946570920035,
 'mean': -0.046451146424399035,
 'p05': -0.2157462459226021,
 'p95': 0.13359881870812104,
 'VaR_5': -0.2157462459226021,
 'CVaR_5': -0.2541175788322952}

What this result means (10 trading days ≈ ~2 weeks)

These are cumulative log-returns over the next ~2 weeks, simulated from your conditional distribution (features held constant).

1) Typical outcome (median)

median = −0.0483

Interpretation:

In a “typical” scenario, the model expects about a −4.8% move in log terms over the next 2 weeks.

Converted to simple return:

exp
⁡
(
−
0.0483
)
−
1
≈
−
4.7
%
exp(−0.0483)−1≈−4.7%

So: ~−4.7% typical 2-week return.

2) Upside / downside uncertainty (p05 and p95)

p05 = −0.2157

p95 = +0.1336

Interpretation:

90% of simulated outcomes fall between about −21.6% and +13.4% (log terms).

Converted to simple returns:

So:

downside could be around −19% in a bad 2-week period

upside could be around +14% in a strong 2-week period

This wide band is very “crypto-realistic.”

3) Risk metrics (VaR and CVaR)

VaR(5%) = −0.2157

CVaR(5%) = −0.2541

Interpretation:

VaR(5%):

“There is a 5% chance the 2-week return is worse than about −21.6% (log).”

CVaR(5%):

“If we are already in the worst 5% of outcomes, the average loss is about −25.4% (log).”

Converted to simple returns:

So:

Worst 5% threshold ≈ −19%

Average of worst tail ≈ −22%

That’s a meaningful tail-risk estimate.

Quick sanity check: why VaR equals p05 here

Because VaR(5%) is the 5th percentile of the simulated horizon return distribution. So VaR_5 and p05 should match (or be extremely close). ✅

#### explicit start/end dates (end_date)

Pick a date range you care about. Example: “from the last date to 2 months later”.

In [13]:
out_range = hs.forecast_horizon(
    bundle=bundle,
    features_df=features_df,
    start_date=start_date,
    end_date="2026-04-30",   # choose any future date you want
    n_scenarios=5000,
    alpha=0.05,
    seed=42,
)

out_range.horizon_days, out_range.summary


(58,
 {'median': -0.2749376154662766,
  'mean': -0.2685781624696419,
  'p05': -0.6596595503163974,
  'p95': 0.15168541066326172,
  'VaR_5': -0.6596595503163974,
  'CVaR_5': -0.7582083865658462})

“years” using horizon_days

Rough trading-day approximations:

1 year ≈ 252 trading days

2 years ≈ 504

5 years ≈ 1260

In [14]:
out_1y = hs.forecast_horizon(
    bundle=bundle,
    features_df=features_df,
    start_date=start_date,
    horizon_days=252,
    n_scenarios=5000,
    alpha=0.05,
    seed=42,
)

out_1y.summary


{'median': -1.1934690456559065,
 'mean': -1.1837991971312374,
 'p05': -2.0189292511211536,
 'p95': -0.31847797742864653,
 'VaR_5': -2.0189292511211536,
 'CVaR_5': -2.2240692404690043}

#### convert log-return to simple return (%)

Log returns are additive (good for modeling), but users like % gain/loss.

In [15]:
import numpy as np

def log_to_simple(x):
    return np.exp(x) - 1

def summarize_simple(out):
    s = out.summary
    return {
        "horizon_days": out.horizon_days,
        "median_%": float(log_to_simple(s["median"]) * 100),
        "p05_%": float(log_to_simple(s["p05"]) * 100),
        "p95_%": float(log_to_simple(s["p95"]) * 100),
        "VaR5_%": float(log_to_simple(s["VaR_5"]) * 100),
        "CVaR5_%": float(log_to_simple(s["CVaR_5"]) * 100),
    }

summarize_simple(out_2w), summarize_simple(out_range), summarize_simple(out_1y)


({'horizon_days': 10,
  'median_%': -4.7123015193136,
  'p05_%': -19.40602135650682,
  'p95_%': 14.293420383632904,
  'VaR5_%': -19.40602135650682,
  'CVaR5_%': -22.439939752815473},
 {'horizon_days': 58,
  'median_%': -24.038048974420136,
  'p05_%': -48.2972673678522,
  'p95_%': 16.37940616358391,
  'VaR5_%': -48.2972673678522,
  'CVaR5_%': -53.149494451205804},
 {'horizon_days': 252,
  'median_%': -69.6832264961154,
  'p05_%': -86.72024181572668,
  'p95_%': -27.274490619358005,
  'VaR5_%': -86.72024181572668,
  'CVaR5_%': -89.18319497606386})

In [21]:
import joblib

joblib.dump(bundle, "artifacts/models/BTC_quantile_bundle.joblib")


['artifacts/models/BTC_quantile_bundle.joblib']

In [20]:
from pathlib import Path

Path("artifacts/models").mkdir(parents=True, exist_ok=True)
print("artifacts/models directory ensured")


artifacts/models directory ensured
